In [0]:
from pyspark.sql import functions as F

df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load('/Volumes/workspace/jamal/olist/olist_customers_dataset.csv')

# Add audit columns 
df_bronze = df_raw \
    .withColumn("_ingested_at", F.current_timestamp()) \
    .withColumn("_source_file", F.lit("olist_customers_dataset.csv"))

# ── Step 3: Write to Delta (overwrite for idempotency on free tier)
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.jamal.bronze_customers")

# ── Step 4: Verify
display(spark.sql("SELECT * FROM workspace.jamal.bronze_customers LIMIT 5"))
spark.sql("SELECT COUNT(*) as total_rows FROM workspace.jamal.bronze_customers").show()